In [1]:
import os
import pandas as pd
import gc
from joblib import Parallel, delayed
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity
import unicodedata
import re
import random
from itertools import chain
from keybert import KeyBERT
from tqdm import tqdm
from collections import defaultdict
import logging
import torch
import openai
from nltk.corpus import wordnet
from itertools import product
import nltk
nltk.download('wordnet')


[nltk_data] Downloading package wordnet to /Users/mgor/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
### DataFrame and Graphs Settings
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colum', None)
pd.set_option("display.max_colwidth", None)



In [13]:
# Dictionary definitions
pecuniary_benefits = {
    "stock options or equity grants": [
        "stock options", "employee stock ownership", "equity grants",'stock purchase program',
        "company shares", "esop", "share options", "long term incentives",
        "equity packages", "stock purchase plan", "employee stock",
        "restricted stock units", "stock appreciation rights",
        "performance based equity", "employee share purchase plans","employee owned"
    ],
    "retirement plans": [
        "401k", "retirement", "pension", "savings plans",
        "employer matched retirement", "403b", "rrsp", "superannuation",
        "retirement benefits", "defined benefit plan", "defined contribution plan",
        "annuities", "deferred compensation plans",'savings account','savings programs',
        'savings stock investment plans'
    ],
    "insurance benefits": [
        "health insurance", "dental insurance", "vision insurance","healthcare insurance","annuity insurance",
        "life insurance", "disability insurance", "medical plans","dental services","dental care",
        "mental health coverage", "pet insurance", "dependent insurance","disability protected","plan care",
        "dependent care", "dependent coverage", "family insurance","disability need","longterm disability"
        "spouse insurance", "critical illness insurance", "accident coverage","dental program",'physical disability', 
        "long term disability", "short term disability","disability term","disability status","medical dental",
        "vision care plans", "long term care insurance", "travel insurance","mental health care","life disability",
        "emergency childcare reimbursement","vision coverage","supplemental insurance","health vision",'mental disability'
    ],
    "tuition reimbursement or educational assistance": [
        "tuition reimbursement", "education assistance", "scholarships","education benefits",
        "career training sponsorship", "certification reimbursement","school loans",
        "student loan assistance", "education support", "tuition aid",'tuition discounts reimbursement',
        "dependent scholarship", "college scholarship", "educational programs",'tuition coverage',
        "work study support", "dependent tuition","tuition benefit",'student loan repayment assistance',
        'sponsorship opportunities','reimbursements for licenses and certifications',
        'scholarship programs', 'semester reimbursement program',
        'tuition assistance',
        'tuition fee','assistance scholarship programs',
        'scholarship program'
    ],
    "commuter subsidies": [
        "commuter benefits", "transportation stipend", "mileage reimbursement",
        "parking allowance", "subsidized transportation", "company shuttle","home transportation",
        "public transit reimbursement", "commuter allowance", "carpool support",'parking reserved',
        'obtain parking',
        'travel allowance','travel expense',
        'possible transportation',
        'transportation shuttle',
        'parking employee','parking provided',
        'reimbursements travel',
        'parking access',
        "public transportation", "electric vehicle subsidies", "bike to work schemes","mileage calculations expenses"
    ],
    "paid time off": [
        "paid time off", "pto", "vacation days", "sick leave",
        "parental leave", "maternity leave", "bereavement leave",'paternity leave','paid maternity',
        "paid holidays", "flexible time off", "holiday pay", "personal leave",'sick time',
        "wellness days", "mental health days", "vacation time", "sabbaticals",
        "floating holidays", "personal days", "volunteer time off", "vacation",'paid time holidays',
        'leave paid time','holiday bonus',
        'maternity paternity leave','parental bonding leave',
        'generous holiday',
        'protected pregnancy maternity',
        "maternal leave", "paternal leave"
    ],
    "additional benefits": [
        "bonus", "child support", "childcare", "competitive benefits","child care",
        "employee assistance program", "employee discount", 'benefits including comprehensive',
        'excellent benefits including','benefits offered','money discounts'
        'personal time','bereavement paid',
        'paid bereavement',
        'benefits paid','exceptional benefits','exemplary compensation benefits package',
        'competitive compensation benefits','relocation package available',
        'care flex spending account', 'adoption fertility assistance',
        'care spending account', 'day care','coverage adoption fertility',
        "flexible schedule", "full benefits", "guaranteed hours", 'employee referral award',
        "wellness", "union", 'offer great benefits','adoption benefit','family assistance program',
        "adoption assistance", "fertility benefits", "surrogacy assistance",
        "gym memberships", "wellness stipends", "legal assistance plans",
        "home office stipends", "relocation assistance",'competitive compensation benefits',
        "dependent care","flexible spending accounts","comprehensive benefits","outstanding benefits",
        "relocation provided","company discounts","wellness benefits",
    ],
    "profit sharing": [
        "profit sharing", "company wide profit sharing programs","share value","value sharing",'profit ownership'
    ]
}


job_design_characteristics = {
    "flexible work arrangements": [
        "remote work", "work from home", "hybrid schedule",
        "flexible hours", "flextime", "compressed workweek",'home job',
        "part time options", "job sharing", "flexible shifts","work remotely",
        "adjustable hours", "distributed team", "remote friendly",
        "flexible workplace", "work anywhere", "virtual work environment",
        "remote friendly culture", "location independent","working remote",
        "telecommuting options", "global teams", "asynchronous work","flexible work schedule",
        "work anywhere policy","remote management","flexible schedule"
    ],
    "job autonomy": [
        "autonomy", "decision making authority", "independence",
        "self directed", "minimal supervision", "creative freedom",
        "control over work", "self managed", "initiative driven",
        "ownership of projects", "empowered decision making",
        "self starter", "independent decision making", 
        "flat hierarchy", "flat organization", "non hierarchical structure",
        "decentralized decision making", "empowered teams",
        "independent work environment", "project ownership",
        "freedom to innovate", "minimal oversight",
        "decision making flexibility","work independently",
        "flat structure", "collaborative decision making","decentralized management",
    ],
    "work life balance initiatives": [
        "work life balance", "worklife balance", "worklife program", 
        "workfamily", "healthy worklife", "sustainable worklife", 
        "balanced life", "balanced lifestyle", "family time", 
        "healthier and happier", "time for family", 'balance work personal lives',
        "balance work and leisure", "healthy lifestyle", 'balance personal',
        "work life harmony", "work life integration", 'worklife quality',
        "family friendly policies", "flexible leave policies",'healthy environment',
        "wellness programs", "mindfulness support", 'familyfriendly hours',
        'family healthy','career support worklife',
        'programs help employees healthy','celebrate family',
        "reduced work hours", "part time schedules", 'healthy habits',
        "employee well being initiatives", "family leave policies","comprehensive leave program"
        "mental health days","family medical leave"
    ],
    "secure and decent work": [
        "secure work", "decent work", "stable employment",
        "ethical work environment", 'safe healthy environment',
        'safe working environment','safety practices',
        'supportive corporate environment',
        "long term opportunities", "job stability",'safe inclusive environment',
        "livable wages", "accident safety investigations",'safe healthful working',
        "iso standards", "iso 27001","fair ethical manner","stable company","ensure safe work",
        "stable workforce",'safety practices','performing tasks safely','work safety', 'safe work environment',
        'career opportunities stability','safe healthful working',
        'secure job',
        'safe work practices',
        'safe productive work',
        'adequate work',
        'career stability',
        'safety regulations',
        'safety policies',
        'safety concerns',
        'safety rules',
        'safety precautions',
        'safety hazards',
        'job security',
        'stable supportive corporate',
    ]
}




career_development_opportunities = {
    "skill development programs": [
        "training programs", "skill development","paid continuing education",
        "conference participation", "career growth workshops",'development training',
        "online courses", "professional development", "upskilling programs",
        "reskilling opportunities", "sponsor industry certifications", 
        "continuous learning", "worker development", "training initiatives"
        "jumpstart your career", "looking to grow","employee training",
        "technical training", "skills training", "learning programs","educational opportunities",
        "specialized training", "leadership training", "career training","continuing education reimbursement",
        "career development workshops", "skills enhancement","great training","management training",
        
    ],
    "mentorship and coaching": [
        "mentorship", "coaching", "career guidance", "employee development",
        "mentor programs", "one on one coaching", "peer mentorship",
        "executive coaching", "leadership mentoring",'candid meaningful feedback',
        "career coaching", "mentoring programs", "guidance programs","consistent feedback",
        "development coaching", "personalized coaching","feedback constantly","performance feedback",
        "personal growth","learning opportunities",'career advice','performance evaluations','career coaches',
        'regular feedback'
    ],
    "clear career progression paths": [
        "promotion tracks", "career progression", "advancement opportunities", 'opportunities grow',
        "opportunities for advancement", "opportunities for growth", "accelerate career",
        "growth opportunities", "career advancement", "career opportunities", 'internal promotions',
        "career mobility", "career development", "dynamic careers", "promotion cycle",'climb ladders',
        "careerdriven individuals", "rapid advancement", "career path","pursue desired career",
        "promotion opportunities", "climbing the ladder", "advance career","personal advancement",
        "growth tracks", "future leadership roles", "development opportunities",'growth plan', 'offers promotions',
        "clear advancement paths", "progression plans","grow career","promotion potential"
    ],
    "support for lateral moves": [
        "skill diversification", "internal transfers", "cross functional moves",
        "lateral moves", "cross training opportunities", "internal mobility",
        "role flexibility", "career transitions within company", 
        "careeradvancing", "career growth", "professional growth",
        "role switching", "internal job opportunities", 
        "cross departmental opportunities", "team transitions", 
        "job rotation programs", "diverse career paths"
    ]
}



organizational_culture = {
    "psychological safety": [
        "open communication", "respectful conflict resolution", 
        "constructive feedback", "safe to voice opinions", 
        "no fear of retaliation", "supportive environment", 
        "encouraging expression", "trust in leadership", 
        "employees voice", "workers voice", "employee ideas", 
        "warm and friendly culture", "trust", "protected",
        "inclusive feedback", "safe environment", "psychologically safe",
        "employee well being", "non judgmental"
    ],
    "inclusion of diverse perspectives": [
        "diversity in decision making", "inclusive leadership",
        "representation in leadership", "valuing diverse opinions",
        "inclusive workplace", "cultural inclusivity", "multicultural environment",
        "diverse voices", "equitable decision making", "inclusive teams","inclusive work environment",
        "diverse perspectives", "multicultural workforce", "cross cultural inclusion"
    ],
    "recognition and celebration": [
        "employee recognition", "celebrating achievements", 
        "shout outs", "employee awards", "appreciation programs",'incentive awards',
        'demonstrated achievements','valued employees',
        'incentive program',
        'annual incentive',
        'performancebased incentives',
        'employee earnings','incentive pay',
        'merit bonuses',
        'employee rewards recognition',
        'staff appreciation','celebrates success',
        "team celebrations", "milestone recognition", "achievement rewards", 
        "recognized and rewarded", "reinvest in our employees", 
        "promote from within", "promoting from within","recognized rewarded",
        "peer recognition", "employee appreciation", 
        "team recognition", "achievement celebrations", "employee value"
    ],
    "collaborative environment": [
        "team collaboration", "supportive teams", "peer support",'team effort', 
        "cross functional collaboration", "helpful colleagues",'cooperative harmonious working',
        "team oriented environment", "teamwork culture", "collaborative atmosphere", 
        "team environment", "teamwork", "employee involvement", 'team culture','collaborative open environment',
        "employee relations", "worker representation", "teamoriented environment",
        "working men and women", "employees are key",  'supportive appreciative team','work collaboratively',
        "employees are our greatest asset", "care of our people",'teambased culture',
        "collective problem solving", "team building", "collaborative work environment",
        "collaborative teams", "shared goals", "cooperative workplace",'responsible employers'
    ],
    "transparency in leadership": [
        "transparent leadership", "clear communication from leaders",
        "honest decision making", "accessible management","ethical leadership",
        "open door policy", "leader transparency", "managerial openness",
        "transparency", "leadership integrity", "honest leadership",   'positive leadership',
        'organizational accountability','dynamic leadership',
        'visible leadership',
        'organizational communication',
        'environment openness',
        'held accountable','strong leadership',
        'leadership communication',
        'management clear communication',
        'leadership oversight',
        "transparent decision making", "accountable leadership", "openness in management"
    ],
    "workplace recognition": [
        "best companies work", "great place work", 'award winning staff','best large employers',
        "employee friendly workplace", "award winning workplace", "best places work","best employer", 
        "recognized employer", "employer of choice", "best places to work",'perfect place work',
        'wonderful place work','ranked best companies','awardwinning company','company ranking',
        'best midsize employers','leading global supplier','leading manufacturer','leading partner',
        'excellent place work','worlds admired companies','leading companies world', 'awesome place work',
        'large companies world','100 best companies','awardwinning fastgrowing company'
    ]
}


meaningful_work = {
    "alignment with societal values": [
        "mission driven", "socially conscious", "ethical goals", 'company mission',
        "values alignment", "purpose driven", "social impact focus", 'shared objectives values',
        'organizational values','corporate values',
        'company core values',
        "shared values", "corporate mission", "ethical organization",'company values',
        "mission focused roles", "align with the purpose", "share values",
        "meaning of work", "pursuing career with purpose", "mutually rewarding",
        "purpose at work", "life changing", "heart pumping","rewarding career",
        "making impact", "social responsibility", "greater good", "double bottom line"
    ],
    "opportunities to make an impact": [
        "make difference", "positively impact lives", "lasting impact","rewarding work",'better future','improve life','challenging rewarding', 
        'improve quality life','meaningful employment',
        "direct impact roles", 'change lives',"create positive change", "taking toughest challenges",
        "help others", "improve quality of life", "impact driven work", "help people",'fulfilling mission','opportunity make real difference',
        "challenge yourself", "challenging rewarding", 'fulfilling careers','impact peoples lives', 
        "engaging work", "interesting and challenging", "positive impact","helping people",'meaningful work','mission purpose driving',
        "interesting work", "innovative work", "transformative journeys", 'rewarding passion', 'impact peoples',
        "gratifying", "best work of their lives", "work with purpose","make real impact","improving quality life",'rewarding lifelong ',
        "leaving a legacy","career rewarding","improvement human life",'rewarding opportunities'
    ],
    "working on innovation": [
        "breakthrough technologies", "cutting edge research","innovative ideas", "changing future",
        "creative problem solving", "next gen solutions", "worldwide influence",'shaping future','innovative companies',
        "innovative roles", "technology driven innovation", "leading product service innovation",'impact future',
        "pioneering advancements", "disruptive innovation", "world leading companies",'develop breakthrough',
        "designing the future", "worlds most innovative company", 'help build future','help build future',
        "history makers", "partner with great minds", "raise the bar","leading research", 'future constantly changing',
        "creative environment", "free thinkers", "leading innovation",'cutting edge technology','innovative vibrant place work', 
        "transformative technologies", "driving progress","vibrant work environment",'creating better future'
    ],
    "roles tied to solving global challenges": [
        "solving poverty", "climate change solutions", 
        "addressing global challenges", "sustainable development goals", 
        "humanitarian roles", "reduce inequality", "global impact", 'worlds complex challenges',
        'solutions global','worlds challenging issues',"inequality reduction",
        'solve realworld problems',"reducing inequality",
        "environmental challenges", "public health issues", 
        "eradicating hunger", "bright future", "eliminate poverty",
        "a great environment", "environment that inspires",
        "solving global problems", "building resilient societies",
        "advancing human development",'global growth','global security challenges'
    ]
}


environmental_initiatives = {
    "sustainability practices": [
        "sustainability", "carbon neutrality", "net zero", "recycling programs",
        "zero waste", "environmentally friendly", "eco friendly", 
        "green practices", "circular economy", "waste reduction", 
        "sustainable development", "sustainable growth", 
        "sustainable industries", "sustainable packaging", 
        "sustainable practices", "sustainable outcomes", 
        "sustainable urbanization", "double bottom line", 'responsible packaging material',
        'waste policy',
        'sustainable approach','responsible implementation',
        'sustainable ways',
        'sustainable solutions',
        'resources management','sustainable future',
        'sustainable environment','sustainable world',
        'sustained growth',
        "responsible consumption", "responsible farming", 
        "responsible use", "lifestyle of health and sustainability",
        "resource conservation", "reducing waste",'environmentallyfriendly',
        'sustainable infrastructure',
        'sustainable projects',
    ],
    "renewable energy use": [
        "renewable energy", "solar energy", "wind energy", "green energy",
        "clean energy", "solar power", "wind power", "hydro power", 
        "hydrogen power", "alternative energy", "energy efficient", 
        "energy efficiencies", "energy efficiency", "energy star", 'energy performance',
        'sustainable energy',
        'solar strategies',
        "fuel efficiency", "hybrid energy", "hybrid vehicle", "hybrid car", 
        "nuclear", "clean energies", "low carbon energy"
    ],
    "conservation programs": [
        "wildlife protection", "water conservation", "biodiversity programs", 
        "land conservation", "habitat restoration", "forest preservation",
        "water use reduction", "natural resource conservation", 'emission reduction',
        "ecosystem protection", "ocean resources", "overfishing", 
        "fish stock", "amazon rain forest", "bio diversities", "biodiversity", 
        "natural resources", "historic sites", "preservation",'water resources',
        'aquatic resources',
        "protecting habitats", "marine conservation"
    ],
    "climate action": [
        "climate change", "climate crisis", "global warming", 
        "paris agreement", "carbon dioxide", "carbon disclosure", 
        "carbon emission", "carbon neutral", "co2", 'emissions related',
        "greenhouse gas", "environmental impact", 'environmental problems',
        "environmental performance", "environmental footprint", 
        "climate neutral", "impact on the environment",'environmental conditions',
        'environmental production efficiency',
        "climate solutions", "low emissions"
    ],
    "environmental management and practices": [
        "environmental action", "environmental activism", "environmental safety",
        "environmental activities", "environmental disclosure", 
        "environmental management systems", "environmental policies", 
        "environmental policy", "environmental practices", 
        "environmental protection", "environmental reform", 
        "environmental responsibilities", "environmental responsibility", 
        "environmental stewardship", "environmentally inclined", 
        "environmentalist", "material footprint", "material stewardship", 
        "groundwater", "hazardous waste", "toxic chemicals reduction", 
        "waste recycling", "fuel technology", "oxidation", 
        "environmentally safe", "environmentally neutral", 
        "environmental supply chain", "pollution", "pollutant", 
        "polluting", "ozone depletion", "ozone depleting", 
        "gri frameworks", "esg", "gri ratings", "gri standards",
        "clean supply chains", "responsible resource use",'waste disposal',
        'recycling centers',
        'waste spill','environment regulations',
        'recycling nonhazardous solid waste',
        'industry leader recycling',
        'recycling simplified'
    ]
}

social_initiatives = {
    "diversity, equity, and inclusion programs": [
        "aboriginal peoples", "aboriginals", "affirmative action", 
        "african american", "social class", "disabilities", "diversity inclusion",
        "disability status", "disabled", "discriminating", "fair wages",
        "discrimination", "discriminatory", "diversity", "workforce diversity",
        "equal employment", "equal opportunities", "equal opportunity", 
        "equality", "ethnic diversities", "ethnic diversity", 
        "ethnic mosaic", "ethnically", "ethnicities", "ethnicity", 
        "female", "first nations", "first peoples", "gay", "creating inclusive",
        "gender", "hiv", "inclusive", "indigenous", "lesbian", 
        "marital status", "minorities", "minority", "national origin", 
        "native", "pregnancy", "race", "racial", "qualified candidates criminal",
        "religious diversities", "religious diversity", 
        "same sex", "sexual orientation", "underrepresented group", 
        "veteran", "women", "gender inclusion", "diverse leadership",
        "not obligated to disclose", "sealed or expunged", 
        "will not be obligated to disclose", 'fair employment practices',
        "convictions will be reviewed on", 'diverse candidate',
        "convictions will not necessarily bar", 'fairness recruitment',
        "criminal record is not an automatic", 
        "criminal record will be considered",
        "fair hiring practices", 'diverse team',
        'equal pay',
        'pay equal work',
        'diverse backgrounds',
        'diverse group', 'fair equitable compensation',
        'conviction records',
        'diverse workplace',
        'diverse workforce',
        'diverse environment',
        'culturally diverse',
        'diverse friendly','applicants receive equal consideration', 
        'diverse community','equalopportunity employer', 
        'equal consideration employment'
    ],
    "community engagement": [
        "community engagement", "volunteering", "local partnerships", "community involvement","improving communities",
        "employee volunteer programs", "neighborhood support", 'volunteer hours','shared responsibility',
        "grassroots initiatives", "community development", "social impact","community building", "build stronger communities",
        "corporate volunteerism", "civic engagement", "community project", "community involvement","strengthening communities",
        "supporting local communities", "socially engaged", "impact communities",'community work','workforce reflecting communities', 
        'engagement opportunities','community efforts','serving community','community initiatives','community initiative'
    ],
    "philanthropy": [
        "charitable giving", "corporate social responsibility", 
        "scholarships", "nonprofit support", "foundation grants",
        "humanitarian aid", "cause based giving", "charity donations",
        "philanthropic initiatives", "donations", "giving back",'corporate giving',
        "community philanthropy", "donor support",'responsible corporate','community giving'
    ],
    "ethical supply chains": [
        "ethical sourcing", "fair trade", "responsible sourcing", 
        "supply chain transparency", "labor rights", "sustainable sourcing", 
        "supplier diversity", "worker protections", "fair labor standards",
        "environmental supply chain", "responsible farming", 
        "worker equity", "transparent supply chains",'public disclosure',
    ]
}


job_tasks_requirements = {
    "job_tasks": [
        "manage", "lead", "design", "develop", "coordinate", "plan", "analyze",
        "implement", "research", "organize", "support", "communicate", "monitor",
        "evaluate", "train", "facilitate", "present", "report", "maintain",
        "oversee", "test", "consult", "build", "solve", "write", "debug",
        "deploy", "execute", "strategize", "coaching", "mentoring", "supervising",
        "duties include", "responsibilities include", "tasked with", "essential responsibilities"
    ],
    "job_requirements": [
        "required", "must", "ability to", "mandatory", "minimum qualifications",
        "experience in", "desired", "preferred", "essential", "proficiency in",
        "certification", "background in", "knowledge of", "expertise in",
        "competency in", "fluency in", "responsibilities include", "ability to manage",
        "strong understanding of", "track record of", "commitment to", "ability to lead"
    ],
    "qualifications": [
        "bachelor", "master", "phd", "degree", "diploma", "certificate",
        "qualification", "accreditation", "certification", "training",
        "education", "license", "credential", "undergraduate", "postgraduate",
        "graduate", "associate", "specialization", "minor", "major"
    ],
    "experience": [
        "years of experience", "proven track record", "demonstrated experience", "experience demonstrated"
        "prior experience", "work history", "hands-on experience", "relevant experience",
        "extensive experience", "minimum of", "at least", "industry experience",
        "professional experience", "field experience", "internship", "apprenticeship",
        "coaching experience", "mentoring experience", "leadership experience",
        "experience in project management", "experience working in"
    ],
    "skills": [
        "technical skills", "soft skills", "communication skills", "leadership skills",
        "problem-solving", "analytical skills", "teamwork", "collaboration",
        "critical thinking", "time management", "organization", "project management",
        "coding", "programming", "data analysis", "marketing", "sales", "graphic design",
        "presentation skills", "negotiation skills", "customer service", "writing skills",
        "adaptability", "creativity", "decision making", "attention to detail",
        "training skills", "mentoring skills", "facilitation skills","required skills",
        "demonstrates skills assigned"
    ]
}


# Dictionaries
dictionaries = {
    "pecuniary_benefits": pecuniary_benefits,
    "job_design_characteristics": job_design_characteristics,
    "career_development_opportunities": career_development_opportunities,
    "organizational_culture": organizational_culture,
    "meaningful_work": meaningful_work,
    "environmental_initiatives": environmental_initiatives,
    "social_initiatives":social_initiatives,
    "job_tasks_requirements": job_tasks_requirements
}



In [102]:
# Define paths
input_file = "/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Data Aux/US_XML_AddFeed_20101231.parquet"
output_file = "/Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Data Aux/refined_terms_sample_output.parquet"

In [103]:
# Sample size
sample_size = 10
df = pd.read_parquet(input_file)

In [104]:
df.head()

JobID          CanonEmployer     JobDate  \
0  293138603      Kaiser Permanente  2010-12-31   
1  293138606      Kaiser Permanente  2010-12-31   
2  293138615  Georgetown University  2010-12-31   
3  293138618      Kaiser Permanente  2010-12-31   
4  293139350    Delcan Incorporated  2010-12-31   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [10]:
# Text cleaning function
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')
    text = re.sub(r'[\x00-\x1f\x7f-\x9f]', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9\s.,!?]', '', text)
    return re.sub(r'\s+', ' ', text.lower()).strip()

# Load the file and sample data
print(f"Loading data from {input_file}...")
df = pd.read_parquet(input_file)

if 'JobText' not in df.columns or df['JobText'].isnull().all():
    print(f"File {input_file} does not have a valid `JobText` column.")
    exit()

# Sample 10,000 observations
if len(df) > sample_size:
    print(f"Sampling {sample_size} observations...")
    df_sample = df.sample(n=sample_size)
else:
    print(f"Dataset has {len(df)} rows, which is less than the sample size. Using the entire dataset.")
    df_sample = df.copy()
    
df_sample = df_sample[(df_sample['CanonEmployer'].notna()) & (df_sample['CanonEmployer'] != None) & (df_sample['CanonEmployer'] != "None")]
df_sample["CleanedText"] = df_sample["JobText"].apply(clean_text)


Loading data from /Users/mgor/Library/CloudStorage/Dropbox/Second YPP/Data/Data Aux/US_XML_AddFeed_20101231.parquet...
Sampling 10 observations...


In [28]:
# Load models globally
keybert_model = KeyBERT("all-MiniLM-L6-v2")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
summarizer = None  # Set to None initially

def initialize_models():
    """
    Initialize the summarizer model if not already initialized.
    This ensures that the summarizer is loaded properly for each process.
    """
    global summarizer
    if summarizer is None:
        summarizer = pipeline("summarization", model="facebook/bart-large-cnn", tokenizer="facebook/bart-large-cnn")


2024-12-02 18:00:07,209 - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2
2024-12-02 18:00:07,483 - INFO - Use pytorch device: cpu
2024-12-02 18:00:07,486 - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2
2024-12-02 18:00:07,671 - INFO - Use pytorch device: cpu


In [29]:
# Set up logging for debugging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


In [71]:
def refine_dictionaries_with_texts(texts, dictionaries, threshold, ngram_range):
    """
    Refine dictionaries to capture nuanced terms for job postings.
    Ensures:
    1. Retention of distinct terms (e.g., "remote work" and "flexible work arrangements").
    2. Avoidance of terms that lead to double counting (e.g., "remote work" and "remote work arrangement").
    3. Category-aware refinement to prevent redundancy.
    """
    initialize_models()

    # Precompute embeddings for all dictionary terms
    precomputed_embeddings = {}
    existing_terms_set = set()  # Track terms already in the dictionaries
    for dict_name, categories in dictionaries.items():
        precomputed_embeddings[dict_name] = {}
        for category, terms in categories.items():
            terms = [str(term) for term in terms if isinstance(term, str)]
            precomputed_embeddings[dict_name][category] = embedding_model.encode(
                terms, convert_to_tensor=True
            )
            existing_terms_set.update(terms)

    # Extract and clean n-grams
    vectorizer = CountVectorizer(stop_words="english", ngram_range=ngram_range)
    text_features = vectorizer.fit_transform(texts)
    extracted_terms = [term for term in vectorizer.get_feature_names_out() if term.strip()]

    # Compute embeddings for extracted terms
    try:
        extracted_embeddings = embedding_model.encode(extracted_terms, convert_to_tensor=True)
        if extracted_embeddings is None:
            raise ValueError("Extracted embeddings are invalid.")
    except Exception as e:
        raise ValueError(f"Error computing embeddings: {e}")

    # Match terms with dictionary categories
    term_to_best_category = {}
    for dict_name, categories in precomputed_embeddings.items():
        for category, category_embedding in categories.items():
            try:
                similarity_scores = util.pytorch_cos_sim(extracted_embeddings, category_embedding)
                max_similarity, _ = similarity_scores.max(dim=1)
            except Exception as e:
                raise ValueError(f"Error computing similarity for {category}: {e}")

            for idx, similarity in enumerate(max_similarity):
                term = extracted_terms[idx]
                try:
                    similarity = similarity.item()
                    if isinstance(similarity, (int, float)) and similarity >= threshold:
                        if term not in term_to_best_category or term_to_best_category[term][2] < similarity:
                            term_to_best_category[term] = (dict_name, category, similarity)
                except Exception as e:
                    print(f"Error processing similarity for term '{term}': {e}")
                    continue

    # Remove redundancy (prioritize by threshold and length)
    def is_redundant(term, term_list):
        """Check if term is a subset or superset of any term in the term_list."""
        return any(
            term in other_term or other_term in term
            for other_term in term_list
        )

    # Sort terms by threshold first, then length
    term_data = sorted(
        term_to_best_category.items(),
        key=lambda x: (-x[1][2], -len(x[0]))  # Sort by similarity (desc) and length (desc)
    )

    refined_terms = {dict_name: {category: [] for category in categories} for dict_name, categories in dictionaries.items()}
    unique_terms = set()  # Track unique terms added across categories
    for term, (best_dict, best_category, similarity) in term_data:
        if term not in existing_terms_set:  # Skip terms already in the dictionary
            # Avoid subsets/supersets in the same category
            existing_category_terms = refined_terms[best_dict][best_category]
            if not is_redundant(term, existing_category_terms):
                refined_terms[best_dict][best_category].append(term)
                unique_terms.add(term)

    # Final filtering step for dictionary consistency
    for dict_name, categories in refined_terms.items():
        for category, terms in categories.items():
            terms = sorted(terms, key=lambda x: (-len(x), x))  # Sort for consistency
            # Filter against existing terms in the category
            filtered_terms = [
                term for term in terms
                if not is_redundant(term, dictionaries[dict_name][category])
            ]
            refined_terms[dict_name][category] = filtered_terms

    return refined_terms


In [16]:
def parallelize_refine_dictionaries_combined_to_dataframe(df, dictionaries, n_jobs, chunk_size, threshold, ngram_range):
    """
    Parallel processing for refining dictionaries using a combined text approach,
    returning results as a DataFrame.
    
    Parameters:
        df (DataFrame): DataFrame containing the column "CleanedText" with job postings or documents.
        dictionaries (dict): Existing dictionaries to refine.
        n_jobs (int): Number of parallel jobs to run.
        chunk_size (int): Size of chunks for parallel processing.
        threshold (float): Similarity threshold for term matching.
        ngram_range (tuple): Range of n-grams to consider for extraction.

    Returns:
        DataFrame: A DataFrame with columns 'dictionary', 'category', 'proposed_term'.
    """
    try:
        # Combine all text into a single large text
        combined_text = " ".join(df["CleanedText"].dropna().tolist())

        # Split the combined text into chunks
        words = combined_text.split()
        total_words = len(words)
        chunks = [
            " ".join(words[i:i + chunk_size])
            for i in range(0, total_words, chunk_size)
        ]

        # Define a helper function to process each chunk
        def process_chunk(chunk):
            return refine_dictionaries_with_texts(
                texts=[chunk],  # Provide chunk as a single-element list
                dictionaries=dictionaries,
                threshold=threshold,
                ngram_range=ngram_range
            )

        # Perform parallel processing
        results = Parallel(n_jobs=n_jobs, backend='loky', timeout=3600)(
            delayed(process_chunk)(chunk) for chunk in chunks
        )

        # Convert results to a list of rows for the DataFrame
        proposed_terms_list = []
        for result in results:
            for dict_name, categories in result.items():
                for category, terms in categories.items():
                    for term in terms:
                        proposed_terms_list.append({
                            "dictionary": dict_name,
                            "category": category,
                            "proposed_term": term
                        })

        # Create and return the DataFrame
        return pd.DataFrame(proposed_terms_list)

    except Exception as e:
        print(f"Error during parallel execution: {e}")
        return pd.DataFrame(columns=["dictionary", "category", "proposed_term"])


In [73]:
refined_df = parallelize_refine_dictionaries_combined_to_dataframe(
    df_sample,  # DataFrame with a "CleanedText" column
    dictionaries,  # Existing dictionaries
    n_jobs=2,  # Number of parallel jobs
    chunk_size=1000,  # Approximate number of words per chunk
    threshold=0.7,  # Similarity threshold
    ngram_range=(2, 4)  # N-gram range
)

In [74]:
len(refined_df)

87

In [75]:
refined_df = refined_df[refined_df['dictionary'] != 'job_tasks_requirements']

In [102]:
refined_df.category.unique()

array(['retirement plans', 'insurance benefits', 'additional benefits',
       'job autonomy', 'work life balance initiatives',
       'clear role expectations', 'skill development programs',
       'mentorship and coaching', 'clear career progression paths',
       'support for lateral moves', 'inclusion of diverse perspectives',
       'collaborative environment', 'transparency in leadership',
       'sustainability practices',
       'environmental management and practices', 'philanthropy',
       'workplace recognition'], dtype=object)

In [241]:
df = pd.read_parquet('/Users/mgor/Dropbox/Second YPP/Data/Data Aux/refined_terms_single_file.parquet')

In [105]:
openai.api_key = os.environ.get('OPENAI_API_KEY', '')  # Removed hardcoded key


In [209]:
import ast  # For safe evaluation of list-like strings

def evaluate_terms_group_via_api(existing_terms, proposed_terms, category):
    prompt = (
        f"You are evaluating terms for a text analysis dictionary. "
        f"The dictionary is for detecting '{category}' in job postings. "
        f"Here is the list of existing terms in the dictionary: {existing_terms}. "
        f"Here are additional proposed terms: {proposed_terms}. "
        f"Select only the terms from the proposed list that are: "
        f"1. Clear, meaningful, and grammatically correct. "
        f"2. Conceptually relevant to the existing dictionary terms for '{category}'. "
        f"3. Commonly used in professional or HR-related contexts for '{category}'. "
        f"Return only the selected terms as a Python-style list, formatted as: ['term1', 'term2', 'term3']. "
        f"Do not include any explanations or additional text."
    )
    try:
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a helpful assistant for text analysis."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=150,
            temperature=0.1
        )
        response_text = response['choices'][0]['message']['content'].strip()
        # Parse the response into a Python list
        return ast.literal_eval(response_text)
    except Exception as e:
        return []


In [165]:
def create_global_lists_from_dicts(*dictionaries):
    """
    Takes multiple dictionaries and creates global variables for each key,
    containing the associated list of values from the dictionary.
    Spaces in keys are replaced with underscores to create valid Python variable names.

    Args:
        *dictionaries: Dictionaries containing categories as keys and lists as values.
    """
    for dictionary in dictionaries:
        globals().update({key.replace(" ", "_"): value for key, value in dictionary.items()})

In [195]:
def create_independent_lists(dataframe, group_column, value_column):
    """
    Groups the values of a column in a DataFrame by a specified group column
    and dynamically creates independent lists named after each group,
    appending "_proposed" to the list name. Prints all created lists.

    Args:
        dataframe (pd.DataFrame): The input DataFrame.
        group_column (str): The column to group by.
        value_column (str): The column whose values will be collected into lists.
    """
    grouped = dataframe.groupby(group_column)[value_column].apply(list)
    
    created_lists = {}  # Dictionary to store created lists for verification
    
    for group_name, items in grouped.items():
        # Replace spaces with underscores and create list names
        list_name = f"{group_name.replace(' ', '_')}_proposed"
        globals()[list_name] = items
        created_lists[list_name] = items  # Store for printing
        

In [250]:
create_global_lists_from_dicts(pecuniary_benefits, job_design_characteristics, career_development_opportunities,organizational_culture, meaningful_work, environmental_initiatives,social_initiatives,job_tasks_requirements)

In [243]:
create_independent_lists(df, "category", "proposed_term")

Created Lists:
additional_benefits_proposed: ['selfdetermined schedule', 'benefits flexible', 'potential benefit', 'discount sales', 'benefit best', 'benefits starting', 'pay benefits', 'benefits package offer', 'manages schedule', 'schedule based', 'flexible scheduling', 'discounts free', 'schedule management', 'planning relocate', 'benefit summary', 'benefits health', 'schedule hour', 'competitive wages paid', 'flexible scheduling', 'scheduling software', 'assistance company', 'awesome benefit', 'benefits short', 'programs available employees', 'discounts things like', 'pay gym', 'options benefits', 'benefits program helps create', 'scheduling duties', 'salary benefits', 'benefits help', 'best benefits', 'benefits deserve', 'schedule hour', 'pay benefits', 'schedule want expand', 'benefits flexible', 'stipend optimize home office', 'scheduling experience', 'specific scheduling', 'adoption newer', 'plus benefits', 'scheduling objectives', 'routine scheduling', 'timely scheduling', 'pa

In [244]:
# Process each unique category in the dataframe
for category in df["category"].unique():
    existing_terms = globals().get(category.replace(" ", "_"), [])
    proposed_terms = globals().get(f"{category.replace(' ', '_')}_proposed", [])
    if existing_terms and proposed_terms:
        # Call the API for each category
        api_terms = evaluate_terms_group_via_api(existing_terms, proposed_terms, category)
        # Save the API result with a dynamic variable name
        globals()[f"{category.replace(' ', '_')}_api"] = api_terms

# Initialize a dictionary to store non-empty _api lists as columns
api_results = {}

# Process each unique category
for category in df["category"].unique():
    api_list_name = f"{category.replace(' ', '_')}_api"
    api_terms = globals().get(api_list_name, [])
    if api_terms:  # Include only non-empty lists
        # Add the category as a column, with its terms as the values
        api_results[category] = api_terms

# Convert the dictionary into a DataFrame
api_df = pd.DataFrame.from_dict(api_results, orient="index").transpose()


In [245]:
api_dict = api_df.to_dict(orient='list')

In [256]:
def generate_similar_terms_list_fixed_length(terms):
    def get_synonyms(word):
        synonyms = set()
        for syn in wordnet.synsets(word):
            for lemma in syn.lemmas():
                synonyms.add(lemma.name().replace("_", " "))  # Convert underscores to spaces
        return synonyms if synonyms else {word}  # Include the original word if no synonyms found

    results = {}

    for term in terms:
        words = term.split()
        synonym_groups = [get_synonyms(word) for word in words]

        # Generate combinations of synonyms, keeping the length constant
        similar_terms = set(" ".join(combo) for combo in product(*synonym_groups) if len(combo) == len(words))
        results[term] = similar_terms

    return results


In [251]:
stock_options_or_equity_grants

['stock options',
 'employee stock ownership',
 'equity grants',
 'stock purchase program',
 'company shares',
 'esop',
 'share options',
 'long term incentives',
 'equity packages',
 'stock purchase plan',
 'employee stock',
 'restricted stock units',
 'stock appreciation rights',
 'performance based equity',
 'employee share purchase plans',
 'employee owned']

In [257]:
similar_terms_dict = generate_similar_terms_list(additional_benefits)


In [30]:
# Text cleaning function
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')
    text = re.sub(r'[\x00-\x1f\x7f-\x9f]', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9\s.,!?]', '', text)
    return re.sub(r'\s+', ' ', text.lower()).strip()

# Define function to count keywords and list matched terms
def keyword_count_and_terms(text, dictionary):
    count = 0
    terms = []
    for category, keywords in dictionary.items():
        for keyword in keywords:
            if keyword in text:
                count += 1
                terms.append(keyword)
    return count, terms

# Load and clean the data
def process_file(df):
    df = df.dropna(subset = ['JobText'])
    df= df[(df['CanonEmployer'].notna()) & (df['CanonEmployer'] != None) & (df['CanonEmployer'] != "None")]
    df['CleanText'] = df['JobText'].apply(clean_text)
    #df = df.drop(columns=['IsDuplicate','IsDuplicateOf', 'JobReferenceID'])

    # Dictionary counts
    for dict_name, dictionary in dictionaries.items():
        df[f'{dict_name}_count'], df[f'{dict_name}_terms'] = zip(
            *df['CleanText'].apply(lambda x: keyword_count_and_terms(x, dictionary))
        )
    df['main'] = df['CleanText'].apply(lambda x: keyword_count_and_terms(x, old)[0])
    # Date
    df['year'] = df.JobDate.apply(lambda x: x[0:4])
    df['month'] = df.JobDate.apply(lambda x: x[5:7])
    
    return df

### Paralellize function
def parallelize_dataframe(df, func, n_cores=3):
    df_split = np.array_split(df, n_cores)
    pool = Pool(n_cores)
    df = pd.concat(pool.map(func, df_split))
    pool.close()
    pool.join()
    return df


In [16]:
### Import packages
import pandas as pd
import os
import re
import unicodedata
#from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import PCA
import numpy as np
import datetime
from multiprocessing import Pool

In [275]:
pip install bertopic

     |████████████████████████████████| 158 kB 1.5 MB/s eta 0:00:01
     |████████████████████████████████| 6.9 MB 833 kB/s eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
    Preparing wheel metadata ... done
     |████████████████████████████████| 15.6 MB 1.1 MB/s eta 0:00:01
     |████████████████████████████████| 88 kB 1.5 MB/s eta 0:00:01
     |████████████████████████████████| 302 kB 1.4 MB/s eta 0:00:01
     |████████████████████████████████| 2.4 MB 906 kB/s eta 0:00:01
     |████████████████████████████████| 56 kB 948 kB/s eta 0:00:01
     |████████████████████████████████| 25.5 MB 1.2 MB/s eta 0:00:01
  Using cached importlib_metadata-6.7.0-py3-none-any.whl (22 kB)
  Created wheel for hdbscan: filename=hdbscan-0.8.40-cp37-cp37m-macosx_10_9_x86_64.whl size=837103 sha256=93db98ea254fb0a2dab63076e56eff5d46d8a1462c388b382d36a0465a8ff213
  Stored in directory: /Users/mgor/Library/Caches/pip/wheels/4e/b1/06/085256a906c079ddca6cc382

In [19]:
df_split = np.array_split(df, 2)
for i, chunk in enumerate(df_split):
    print(f"Chunk {i}: {chunk.shape}")

Chunk 0: (14082, 7)
Chunk 1: (14082, 7)


In [20]:
df_aux = df.iloc[i * (100000):(100000) * (i + 1) - 1]
print(f"Chunk {i}: Shape {df_aux.shape}")

Chunk 1: Shape (0, 7)


In [24]:
train = pd.DataFrame()

In [31]:
df_aux=df[0:100]
aux=process_file(df_aux)

In [44]:
df_aux=deserialize_complex_columns(df_aux)

In [48]:
df_aux.pecuniary_benefits_terms[0]

b'["bonus"]'

In [50]:
decoded_data = json.loads(df_aux.decode('utf-8'))

In [51]:
decoded_data

['bonus']

In [95]:
import json

def process_terms_columns(df):
    """
    Decode and deserialize JSON-like strings in byte strings for columns ending with '_terms'.
    """
    for col in df.columns:
        if col.endswith('_terms'):  # Check if column name ends with '_terms'
            df[col] = df[col].apply(
                lambda x: json.loads(x.decode('utf-8')) if isinstance(x, bytes) else x
            )
    return df


In [98]:
decoded_data = process_terms_columns(df_aux)

In [85]:
decoded_data = json.loads(df_aux.pecuniary_benefits_terms[0].decode('utf-8'))

In [86]:
decoded_data

['bonus']

In [97]:
decoded_data.head()

JobID                         CanonEmployer     JobDate  \
0  292318296                           Wells Fargo  2010-12-01   
1  292318303                                   Djo  2010-12-01   
2  292318307                 Banfield Pet Hospital  2010-12-01   
3  292318309                  University of Kansas  2010-12-01   
4  292318292  American Cancer Society Incorporated  2010-12-01   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

/opt/anaconda3/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3331: DtypeWarning: Columns (1) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)
